# 01 · Data audit

> ⚕️ Educational research prototype on a **synthetic** cohort — not for clinical use.

In [1]:
import pandas as pd

from medpredict.common.config import get_settings
from medpredict.data.cohort import data_audit, load_cohort

pd.set_option("display.width", 200)
s = get_settings()
df = load_cohort()
a = data_audit(df)
{k: a[k] for k in ("n_rows", "n_patients", "prevalence", "missingness", "leakage_columns_present", "prevalence_by_site")}

{'n_rows': 11001, 'n_patients': 6000, 'prevalence': 0.27824743205163166, 'missingness': {'hba1c': 0.11744386873920552, 'creatinine': 0.02863376056722116}, 'leakage_columns_present': ['followup_calls_30d', 'readmission_cost'], 'prevalence_by_site': {'A': 0.2451, 'B': 0.263, 'C': 0.3814}}

## Target leakage check
`followup_calls_30d` and `readmission_cost` are only known *after* the outcome window.

In [2]:
df.groupby("readmit_30d")[["followup_calls_30d", "readmission_cost"]].mean().round(2)

,followup_calls_30d,readmission_cost
readmit_30d,,
0,0.5,0.00
1,3.0,9063.04


Both are near-perfect predictors — and therefore unusable. They are listed in `configs/target.yaml`
and a unit test fails if they ever appear in the feature list.

In [3]:
df.groupby("site").hba1c.apply(lambda x: x.isna().mean()).round(3)

site
A    0.053
B    0.050
C    0.384
Name: hba1c, dtype: float64